# 03a. 네 가지 흐름 ①: 순차와 병렬

앞선 02 노트북에서는 Supervisor와 Handoff 구조를 직접 실행하고 비교했습니다. 이름이 서로 다른 여러 Multi-Agent 아키텍처를 살펴보았지만, 실제 실행 단계에서 그래프가 수행할 수 있는 기본 제어 흐름은 네 가지뿐입니다. 앞선 작업이 끝나야 다음 작업으로 넘어가거나(**순차**), 여러 작업을 동시에 실행하거나(**병렬**), 상태 조건에 따라 경로를 선택하거나(**분기**), 기준을 만족할 때까지 이전 단계로 되돌아갑니다(**루프**). 따라서 이번 단계에서는 화려한 패턴 이름을 잠시 내려놓고 이 네 가지 기본 흐름을 직접 구현한 뒤, 다시 패턴 이름들을 네 가지 흐름의 조합으로 번역해 봅니다.

이번 **03a** 노트북에서는 네 가지 흐름 중 **업무의 선행 조건만으로 결정되는 두 가지 흐름인 순차와 병렬**을 다룹니다. 실행 도중의 결과값을 보고 다음 경로를 결정하는 분기와 루프, 그리고 아키텍처 패턴 이름의 번역은 이어지는 **03b** 노트북에서 다룹니다.

## 학습 목표

- 아키텍처 패턴 이름을 먼저 떠올리는 대신 업무의 상태, 작업, 선행 조건, 종료 조건을 먼저 표로 정의합니다.
- 순차 흐름과 병렬 흐름을 `StateGraph`로 직접 구현하고, 각 구조에서 발생할 수 있는 대표적인 실패를 재현합니다.
- `Send`와 상태 병합 규칙(reducer)의 역할을 이해하고, 작업 분해가 동일한 조건에서만 병렬 실행의 시간 단축 효과를 비교합니다.
- 작업 간 의존 관계를 반영한 최장 경로(longest path) 지연을 계산하고 실제 측정 시간과 대조합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 패턴 말고 업무 먼저 | 7 |
| 2 | 순차와 Context | 7 |
| 3 | 병렬, reducer, 최장 경로 | 12 |
| 4 | 직접 해보기와 정리 | 4 |

## 호출 규모

이 노트북은 모든 LLM 구간에서 **실제 모델**을 호출합니다. `Day03_new/.env` 파일에 `ANTHROPIC_API_KEY`가 설정되어 있지 않으면 첫 번째 코드 셀에서 한국어 안내와 함께 실행이 멈춥니다. 유료 모델을 호출하는 셀은 실행 직전에 예상 호출 규모를 한 줄로 고지하며, 비용은 본인의 API 키로 청구됩니다.

반면 그래프의 연결 구조, reducer 동작, 시간 계산처럼 모델의 추론이 필요 없는 구간은 일반 함수 노드로 실행합니다. 이러한 셀들은 대부분 출력 마지막에 `모델 호출 0회`를 표시합니다. 환경 설정 셀, 그래프를 컴파일하거나 시각화만 하는 셀, 앞선 실행 기록을 표로 출력하기만 하는 셀 역시 별도 문구가 없더라도 모델을 호출하지 않습니다.

| 구분 | 유료 셀 수 | 모델 호출(추정) |
|---|---:|---:|
| 조사 → 작성 → 검토 (역할 분리) | 1 | 5 |
| 실제 Worker 3개의 병렬 실행과 직렬 실행 비교 | 2 | 12~24 |
| 기본 실행 합계 | 3 | 약 17~29 |
| 해보기 3을 활성화할 때 추가되는 호출 | 1 | 4~8 |

각 Worker가 내부에서 조회 도구를 몇 번 호출할지는 모델이 매 실행마다 스스로 판단하므로, 실제 모델 호출 횟수는 실행할 때마다 조금씩 달라질 수 있습니다.

아래 코드는 실습에 필요한 라이브러리를 불러오고 API 키와 기본 고객 요청(T-1001) 데이터를 준비합니다.

In [ ]:
# 표준 라이브러리 불러오기
import json
import operator
import time
from typing import Annotated, Literal, TypedDict

# Deep Agents 및 LangChain/LangGraph 핵심 모듈 불러오기
from deepagents import create_deep_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send
from pydantic import BaseModel

# 실습용 패키지(mas_lab, rag_lab) 모듈 불러오기
from mas_lab.contracts import FinalAnswer, Route, WorkerResult, WorkerTask, join_results, merge_results, plan_hash, validate_final
from mas_lab.desk import Desk, load_billing, load_outage
from mas_lab.hybrid import ROLE_GOALS, ROLES
from mas_lab.runner import make_desk
from mas_lab.structures import OUTCOME_GUIDE, gate_answer, scope_line
from mas_lab.trace import ModelMeter, Trace, longest_path, overlap_seconds, timeline
from mas_lab.view import flows_of, show_graph, show_table, write_design_log
from mas_lab.workers import DENY_WRITES, ROLE_TOOLS, allowlist_middleware, contract_prompt, run_worker
from rag_lab.corpus import load_cases
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

# API 키 확인 및 모델, 고객 요청 사례(T-1001) 초기화
settings = require_key()
model = make_model()
CASES = {case["id"]: case for case in load_cases()}
T1001 = CASES["complex-double-pay"]
desk: Desk = make_desk(T1001, label="점검")
print(key_status(settings))
print("고객 요청 T-1001:", T1001["question"])
print("고객:", desk.scope.entity, "/ 기준일:", desk.scope.as_of, "/ 결제 건수:", len(desk.payments()))
print("측정 장치:", Trace.__name__, ModelMeter.__name__)

## 1절. 패턴 말고 업무 먼저

새로운 자동화 업무를 맡으면 흔히 "Billing Agent, Policy Agent, 그리고 이를 총괄할 Supervisor를 두자"라는 아키텍처 패턴부터 떠올리기 쉽습니다. 하지만 이 문장에는 정작 업무를 처리할 때 **무엇이 먼저 준비되어야 하는지(선행 조건)**에 대한 정보가 전혀 없습니다. 패턴 이름을 먼저 정해 버리면, 이후에는 실제 업무의 자연스러운 흐름을 억지로 그 패턴에 끼워 맞추게 됩니다. 따라서 에이전트 구조를 설계할 때는 항상 아래 다섯 가지 질문을 순서대로 먼저 정의해야 합니다.

1. **목표**: 최종적으로 어떤 상태가 되면 업무가 끝나는가?
2. **필요한 상태**: 그 목표에 도달하려면 State에 어떤 값들이 채워져 있어야 하는가?
3. **작업**: 각 상태 값은 어떤 구체적인 작업을 수행해야 만들어지는가?
4. **선행 조건**: 각 작업은 앞선 어떤 작업이 끝나야만 시작할 수 있는가?
5. **종료 조건**: 각 작업이 정상적으로 끝났다는 것을 무엇으로 판단하는가?

이 다섯 가지를 표로 정리하면 필요한 제어 흐름은 표에서 자연스럽게 도출됩니다. 선행 조건이 걸려 있는 작업들은 **순차**로 연결해야 하고, 서로 선행 조건이 없는 작업들은 **병렬(동시 실행)** 후보가 됩니다.

**먼저 예측하세요.** 고객 요청 T-1001은 "결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."입니다. 아래 코드 셀을 실행하기 전에 필요한 하위 작업들과 각 작업의 선행 작업을 먼저 머릿속으로 그려 보세요. 서비스 크레딧 금액(`credit`)을 계산하려면 어떤 세 가지 조사 결과가 먼저 준비되어야 할까요? 또한 결제 내역 조회(`billing`)와 환불 규정 검색(`policy`) 중 무엇이 먼저 끝나야 할까요, 아니면 실행 순서가 상관없을까요? 예상한 내용을 정리한 뒤 아래 두 코드 셀을 실행하여 확인합니다.

In [ ]:
# T-1001 업무의 하위 작업들과 각 작업의 선행 작업(needs), 채우는 상태 필드(state), 종료 조건(done)을 정의합니다.
TASKS = {
    "verify":   {"needs": [], "state": "customer_verified", "done": "고객이 지원 범위 안인지 확인됨"},
    "billing":  {"needs": ["verify"], "state": "billing_facts", "done": "동일 청구서의 결제 내역이 모두 조회됨"},
    "policy":   {"needs": ["verify"], "state": "policy_evidence", "done": "환불 규정, SLA, 추가 약정의 원문을 찾음"},
    "outage":   {"needs": ["verify"], "state": "outage_facts", "done": "해당 월의 가용률이 조회됨"},
    "credit":   {"needs": ["billing", "policy", "outage"], "state": "credit_amount", "done": "기본 이용료 x 보상 비율을 계산함"},
    "proposal": {"needs": ["credit"], "state": "proposal", "done": "최종 제안과 근거, 불확실성을 작성함"},
    "gate":     {"needs": ["proposal"], "state": "gate_decision", "done": "코드 게이트가 허용, 승인, 거절을 판정함"},
}

def plan_batches(tasks: dict) -> list[list[str]]:
    """위상 정렬(topological sort)을 수행합니다. 각 묶음(batch)은 이전 묶음이 끝나면 동시에 실행할 수 있는 작업 목록입니다."""
    missing = sorted({need for task in tasks.values() for need in task["needs"]} - set(tasks))
    if missing:
        raise ValueError(f"정의되지 않은 작업에 의존합니다: {missing}")
    done, batches = set(), []
    while len(done) < len(tasks):
        ready = sorted(name for name, task in tasks.items() if name not in done and set(task["needs"]) <= done)
        if not ready:
            raise ValueError(f"순환 의존이 발견되었습니다: {sorted(set(tasks) - done)}")
        batches.append(ready)
        done |= set(ready)
    return batches

# 작업 정의 표와 위상 정렬로 묶은 실행 단계(묶음)를 출력합니다.
show_table([{"작업": name, "선행 작업": ", ".join(t["needs"]) or "없음", "채우는 상태 칸": t["state"], "종료 조건": t["done"]} for name, t in TASKS.items()])
for number, batch in enumerate(plan_batches(TASKS), 1):
    print(f"묶음 {number}: {batch}")
print("모델 호출 0회")

In [ ]:
# 실패 재현: JSON 형식은 유효하지만 실제로는 실행할 수 없는 잘못된 계획을 위상 정렬에 넣어 확인합니다.
broken_plans = {
    "순환 의존": {**TASKS, "verify": {**TASKS["verify"], "needs": ["gate"]}},
    "없는 작업 의존": {**TASKS, "credit": {**TASKS["credit"], "needs": ["billing", "tax_lookup"]}},
}
for label, plan in broken_plans.items():
    print(f"{label}: json으로 직렬화됨 = {bool(json.dumps(plan, ensure_ascii=False))}")
    try:
        print("  묶음:", plan_batches(plan))
    except ValueError as error:
        print(f"  거절: {type(error).__name__}: {error}")
print("모델 호출 0회")

**관찰할 것.** 위 작업 표와 묶음 출력을 다시 읽으며 네 가지 기본 흐름이 어디에 자리 잡고 있는지 찾아보세요.

- **순차**: 표에서 `선행 작업` 열이 비어 있지 않은 모든 행입니다. 특히 `credit` 작업은 선행 작업이 `billing`, `policy`, `outage` 세 개라는 점에 주목해 보세요.
- **병렬**: `묶음` 출력에서 한 줄에 두 개 이상의 작업이 함께 묶인 구간(`묶음 2: ['billing', 'outage', 'policy']`)입니다. 이 묶음은 선행 조건상 **"동시에 실행해도 안전한 후보"**라는 뜻이지, 무조건 병렬로 돌려야만 한다는 뜻은 아닙니다.
- **분기**: 표에 별도 열로 적혀 있지는 않지만, `verify`의 종료 조건("고객이 지원 범위 안인지 확인됨")과 `gate`의 종료 조건("허용, 승인, 거절을 판정함")은 그 결과값에 따라 다음 경로가 달라지는 분기 지점입니다.
- **루프**: 현재 표에는 아직 표현되어 있지 않습니다. 만약 어떤 작업의 `종료 조건`이 충족되지 않았을 때 다시 시도하게 만들려면 상태(State)에 어떤 필드가 더 필요할지 생각해 보세요.

두 번째 셀의 실패 재현 출력에서는 `순환 의존`과 `없는 작업 의존` 계획 모두 `json으로 직렬화됨 = True`로 나타났지만 `plan_batches` 함수에서 거절되었습니다. 모델이 계획서를 문법적으로 올바른 JSON이나 Pydantic 스키마로 출력했다는 사실과, 그 계획이 실제로 **실행 가능한(executable) 계획**이라는 사실은 별개입니다. 데이터 형식 검사는 출력 스키마가 담당하고, 논리적 실행 가능성 검사는 위상 정렬과 같은 검증 코드가 담당해야 합니다.

다음 절부터는 이 표의 일부를 실제 `StateGraph` 코드로 옮겨 봅니다.

### 그림으로 보기: T-1001의 선행 조건

앞서 정의한 `TASKS` 표와 `plan_batches`의 묶음 결과를 다이어그램으로 나타내면 아래와 같습니다. 각 화살표는 `needs`(선행 조건)를 뜻하며, 묶음 번호는 선행 조건만으로 계산된 실행 단계입니다.

![T-1001 업무의 선행 조건. verify가 끝나면 billing, policy, outage를 동시에 시작할 수 있고, 셋이 모두 끝나야 credit, proposal, gate가 차례로 이어진다](./assets/03a-1-task-graph.png)

- **묶음 2**에 속한 세 작업(`billing`, `policy`, `outage`) 사이에는 서로를 가리키는 화살표가 없습니다. 서로 상대방의 실행 결과를 필요로 하지 않는다는 뜻이며, 따라서 동시에 실행할 수 있는 후보가 됩니다.
- 반면 **`credit`** 노드로는 세 개의 화살표가 모입니다. 세 선행 작업 중 하나라도 끝나지 않으면 `credit` 계산은 시작할 수 없습니다. 이렇게 앞 단계가 끝나기를 기다려 차례대로 이어지는 구조가 바로 **순차(Sequence)**입니다.

## 2절. 순차와 Context

**문제.** 선행 작업의 결과가 준비되지 않았는데 다음 작업이 먼저 실행되어서는 안 됩니다. 또한 작업들을 단순히 순서대로 연결하는 것만으로는 충분하지 않습니다. 앞선 작업이 만든 결과 중 **무엇을 다음 작업의 입력으로 전달할지** 명확히 정해야 합니다.

LangGraph의 `StateGraph`는 다음 세 가지 핵심 요소로 구성됩니다.

| 요소 | 역할 | T-1001 예시 |
|---|---|---|
| State | 그래프의 모든 작업들이 함께 읽고 쓰는 공유 데이터 구조 | `billing_facts`, `credit` |
| Node | 현재 State를 입력받아 **변경된 필드만** dict로 반환하는 함수 | `verify`, `billing`, `credit` |
| Edge | 한 Node가 끝난 뒤 다음에 어느 Node를 실행할지 연결하는 경로 | `billing → credit` |

여기서 각 Node 함수가 전체 State를 통째로 반환하는 것이 아니라 **자신이 새로 계산해 바꾼 필드만** 반환한다는 점이 매우 중요합니다. 그래프를 `stream(stream_mode="updates")`로 실행하면 각 Node가 반환한 변경 사항만 한 줄씩 차례대로 출력되므로, 실행 단계마다 State가 어떻게 채워지는지 투명하게 추적할 수 있습니다.

아래 코드에서는 순차 흐름의 기본 동작을 먼저 확인하기 위해 크레딧 보상 비율을 `data/business/sla-a-v2.md`의 "99.0% 이상 99.5% 미만" 구간 값인 `15`로 고정해 두었습니다. 이 예제에서는 상태가 순차적으로 전달되는 모양을 관찰하고, 뒤이어 실제 모델이 원문을 읽어 판단하는 구조로 확장합니다.

아래 코드는 `verify → billing → credit`으로 이어지는 순차 그래프를 만들고 단계별 상태 업데이트를 출력합니다.

In [ ]:
# 순차 실행 그래프의 공유 상태(SeqState)와 세 단계 노드를 정의합니다.
class SeqState(TypedDict, total=False):
    entity: str
    customer_verified: bool
    billing_facts: dict
    credit: int

CREDIT_PERCENT = 15  # data/business/sla-a-v2.md의 가용률 99.0% 이상 99.5% 미만 구간 보상 비율(%)

def verify(state):
    # 고객의 결제 원장 존재 여부를 확인하여 customer_verified 필드를 갱신합니다.
    return {"customer_verified": bool(load_billing(state["entity"]))}

def billing(state):
    # 청구서 INV-2026-08의 기본 이용료분을 조회하여 billing_facts 필드를 갱신합니다.
    row = next((p for p in load_billing(state["entity"]) if p["invoice_id"] == "INV-2026-08"), None)
    return {"billing_facts": {"base_fee": row["base_fee_part"] if row else 0}}

def credit(state):
    # 앞선 billing 노드가 채운 base_fee를 읽어 서비스 크레딧 금액을 계산합니다.
    return {"credit": state["billing_facts"]["base_fee"] * CREDIT_PERCENT // 100}

# StateGraph에 노드를 등록하고 고정 간선(add_edge)으로 순차 연결합니다.
builder = StateGraph(SeqState)
for name, node in [("verify", verify), ("billing", billing), ("credit", credit)]:
    builder.add_node(name, node)
builder.add_edge(START, "verify")
builder.add_edge("verify", "billing")
builder.add_edge("billing", "credit")
builder.add_edge("credit", END)
sequence_app = builder.compile()

# 단계별 상태 업데이트(updates)를 스트리밍으로 확인합니다.
for update in sequence_app.stream({"entity": "알파"}, stream_mode="updates"):
    print(update)
print("모델 호출 0회")

### 그림으로 보기: State에 칸이 쌓이는 모습

방금 출력된 `updates` 스트림 세 줄을 State의 필드 변화로 도식화하면 아래와 같습니다. 그림 속 `…` 자리는 실행 시 각 함수가 실제로 계산해 채운 값입니다.

![순차 그래프의 State 변화. 단계마다 Node가 돌려준 칸 하나만 새로 채워지고 앞 단계의 칸은 그대로 이어진다](./assets/03a-2-state-updates.png)

- 각 Node는 State 전체를 덮어쓰는 것이 아니라 **자신이 맡은 필드 하나만** 반환합니다. 따라서 `updates` 출력의 한 줄이 곧 해당 단계가 State에 수행한 변경의 전부입니다.
- 세 번째 `credit` Node는 두 번째 `billing` Node가 State에 기록해 둔 `billing_facts`를 읽어서 계산합니다. 이처럼 앞 단계가 State에 기록한 필드를 다음 단계가 읽는 것이 순차 그래프에서의 **데이터 전달**입니다.

아래 코드에서는 고정 간선(`add_edge`)만 사용했을 때 검증 실패 고객("감마")에게 어떤 문제가 발생하는지 조건부 간선과 비교하여 확인합니다.

In [ ]:
# 실패 재현: 지원 범위 밖의 고객("감마")에게도 고정 간선은 멈추지 않고 다음 단계를 그대로 실행합니다.
def walk(app, initial: dict, **config) -> tuple[list[str], dict]:
    """그래프의 실제 실행 경로(Node 이름 목록)와 최종 상태를 반환합니다. stream 업데이트의 키가 곧 실행된 Node입니다."""
    path, state = [], dict(initial)
    for update in app.stream(initial, stream_mode="updates", config=config or None):
        for node, change in update.items():
            path.append(node)
            state.update(change or {})
    return path, state

# verify 결과(customer_verified)가 거짓이면 즉시 종료(END)하는 조건부 간선 그래프를 만듭니다.
builder = StateGraph(SeqState)
for name, node in [("verify", verify), ("billing", billing), ("credit", credit)]:
    builder.add_node(name, node)
builder.add_edge(START, "verify")
builder.add_conditional_edges("verify", lambda s: "billing" if s["customer_verified"] else END, ["billing", END])
builder.add_edge("billing", "credit")
builder.add_edge("credit", END)
guarded_app = builder.compile()

# 정상 고객("알파")과 미등록 고객("감마")에 대해 고정 간선과 조건부 간선의 실행 경로를 비교합니다.
rows = []
for label, app in [("고정 간선", sequence_app), ("조건부 간선", guarded_app)]:
    for entity in ["알파", "감마"]:
        path, state = walk(app, {"entity": entity})
        rows.append({"그래프": label, "고객": entity, "경로": " → ".join(path), "credit": state.get("credit", "계산 안 함")})
show_table(rows)
print("모델 호출 0회")

이번에는 순차 흐름에서 **역할에 따라 Context를 분리하는 이유**를 살펴봅니다. 만약 자료 조사, 초안 작성, 최종 검토를 하나의 에이전트가 단일 대화 이력으로 모두 처리한다면, 처음에 검색한 방대한 규정 원문 전체가 마지막 답변 작성과 검토 호출까지 계속 따라붙게 됩니다. 순차 파이프라인에서 역할을 나눈다는 것은 단순히 프롬프트 이름을 여러 개 만든다는 뜻이 아니라, **각 단계의 모델 호출에 들어가는 입력(Context)을 필요한 만큼만 남기고 잘라낸다**는 뜻입니다.

| 역할 | 전달받는 입력 | 전달받지 않는 것 | 반환하는 출력 |
|---|---|---|---|
| 조사자 (`researcher`) | 고객 요청, 결제·가용률 데이터, 검색된 규정 원문 | 없음 | 근거 번호가 포함된 핵심 조사 메모 (`notes`) |
| 작성자 (`writer`) | 고객 요청, 조사 메모 (`notes`) | 규정 원문 전체, 조사자의 내부 추론 과정 | 고객에게 전달할 답변 초안 (`draft`) |
| 검토자 (`reviewer`) | 조사 메모 (`notes`), 답변 초안 (`draft`) | 규정 원문 전체 | 통과 또는 반려 판정과 누락·추가 항목 지적 (`review`) |

위 표에서 역할과 역할 사이를 이어 주는 유일한 고리는 **`조사 메모(notes)`**입니다. 만약 이 전달물이 부실하거나 누락되면 뒷단계의 작성자는 정확한 답변을 작성할 방법이 없습니다. 이를 직접 확인하기 위해 동일한 순차 그래프를 **조사 메모를 정상적으로 넘기는 경우**와 **조사 메모를 넘기지 않는 경우**로 나누어 실행해 봅니다. 변인을 정확히 통제하기 위해 두 번째 실행에서는 첫 번째 실행에서 만든 조사 메모를 그대로 재사용하되, 작성자(`writer`)에게만 메모를 전달하지 않고 결과를 비교합니다.

아래 코드는 `조사자 → 작성자 → 검토자`로 이어지는 역할 분리 그래프를 정의하고 두 조건에서 실제 모델을 호출하여 비교합니다.

In [ ]:
# 조사자 → 작성자 → 검토자로 이어지는 순차 역할 그래프와 호출 기록 함수를 정의합니다.
class RoleState(TypedDict, total=False):
    question: str
    sources: str
    notes: str
    pass_notes: bool
    draft: str
    review: str

ROLE_RECORDS = []

def call_role(role: str, prompt: str, run: str) -> str:
    # 역할별 모델 호출의 입력 바이트, 입출력 토큰 수, 소요 시간을 측정하여 기록합니다.
    meter, started = ModelMeter(), time.perf_counter()
    reply = model.invoke(prompt, config={"callbacks": [meter]})
    totals = meter.totals()
    ROLE_RECORDS.append({"실행": run, "역할": role, "입력 바이트": len(prompt.encode()), "입력 토큰": totals["input_tokens"],
                         "출력 토큰": totals["output_tokens"], "시간(초)": round(time.perf_counter() - started, 2)})
    return reply.text

def researcher(state):
    # 이미 생성된 메모(notes)가 있으면 재사용하고, 없으면 원문 자료에서 핵심 사실과 근거 번호를 추출합니다.
    if state.get("notes"):
        return {}
    return {"notes": call_role("조사자", f"조사자입니다. 자료에서 고객 요청에 답하는 데 필요한 사실만 항목으로 뽑고 각 항목 끝에 evidence_id나 결제 번호를 붙이세요. 답변은 쓰지 마세요.\n고객 요청: {state['question']}\n자료: {state['sources']}", "메모 전달")}

def writer(state):
    # pass_notes 설정에 따라 조사 메모를 전달하거나 숨긴 채로 고객 응답 초안을 작성합니다.
    run = "메모 전달" if state["pass_notes"] else "메모 없음"
    notes = state["notes"] if state["pass_notes"] else "전달받은 메모가 없습니다."
    return {"draft": call_role("작성자", f"작성자입니다. 조사 메모의 사실만 써서 고객에게 보낼 답을 다섯 줄 이내로 쓰세요. 메모에 없으면 '확인 필요'로 쓰세요.\n고객 요청: {state['question']}\n조사 메모: {notes}", run)}

def reviewer(state):
    # 작성된 초안을 조사 메모와 대조하여 통과 또는 반려를 판정합니다.
    run = "메모 전달" if state["pass_notes"] else "메모 없음"
    return {"review": call_role("검토자", f"검토자입니다. 첫 줄에 '통과' 또는 '반려'만 쓰고, 다음 줄부터 조사 메모의 사실 중 초안에서 빠진 것과 메모에 없는 문장을 항목으로 쓰세요.\n조사 메모: {state['notes']}\n초안: {state['draft']}", run)}

builder = StateGraph(RoleState)
for name, node in [("researcher", researcher), ("writer", writer), ("reviewer", reviewer)]:
    builder.add_node(name, node)
builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "reviewer")
builder.add_edge("reviewer", END)
role_app = builder.compile()
print("역할 그래프 노드:", [e for e in role_app.get_graph().nodes if not e.startswith("__")])

In [ ]:
# 조사 메모를 작성자에게 전달하는 경우와 전달하지 않는 경우를 차례로 실행하여 초안과 검토 결과를 비교합니다.
announce("조사 → 작성 → 검토, 작성자 메모 유무 비교", model_calls=5)
role_desk = make_desk(T1001, label="역할")
hits = [e for query in ["중복 결제 환불 규정", "월간 가용률 서비스 크레딧 비율", "서비스 크레딧 신청 기한"] for e in role_desk.context.search(query, limit=2)["evidence"]]
sources = json.dumps({"결제": [p for p in role_desk.payments() if p["invoice_id"] == "INV-2026-08"], "가용률": load_outage("알파")["2026-08"],
                      "원문": [{"evidence_id": e["evidence_id"], "quote": e["quote"]} for e in hits]}, ensure_ascii=False)

with_notes = role_app.invoke({"question": T1001["question"], "sources": sources, "pass_notes": True})
without_notes = role_app.invoke({"question": T1001["question"], "sources": sources, "notes": with_notes["notes"], "pass_notes": False})

for label, result in [("메모 전달", with_notes), ("메모 없음", without_notes)]:
    print(f"--- {label} ---\n[초안]\n{result['draft']}\n[검토]\n{result['review']}\n")

In [ ]:
# 역할별 모델 호출 기록(입력 바이트 및 토큰 수)과 원문 대비 조사 메모의 크기를 출력합니다.
show_table(ROLE_RECORDS)
print("원문 자료 바이트:", len(sources.encode()), "/ 조사 메모 바이트:", len(with_notes["notes"].encode()))

**관찰할 것.** 이 실험에서는 에이전트(역할)의 개수가 아니라 역할 사이를 오가는 **전달물(`notes`)의 크기와 품질**을 주의 깊게 살펴봅니다.

- 표 아래에 출력된 `원문 자료 바이트`와 `조사 메모 바이트`를 비교해 보세요. 조사자가 정리한 메모가 원본 자료의 크기를 얼마나 줄였나요? 만약 크기가 거의 줄지 않았거나 오히려 늘어났다면, 이번 역할 분리로 얻은 효과는 토큰 절약이 아니라 단지 역할 책임의 분리뿐입니다. 작성자에게 넘어가는 메모를 더 압축하려면 조사자의 프롬프트 지침에 어떤 제약 조건을 추가해야 할까요?
- `메모 없음` 실행의 `[초안]`을 읽어 보세요. 작성자가 지침을 지켜 '확인 필요'라고 적었다면 근거 없이 추측하지 않고 안전하게 멈춘 것이고, 반대로 구체적인 금액이나 기한을 적었다면 근거 없이 지어낸(환각) 것입니다. 이번 실행에서 어느 쪽으로 나타났는지 확인해 보세요. 어느 쪽이든 작성자 입장에서는 자신에게 어떤 핵심 정보가 누락되었는지 스스로 알 방법이 없습니다.
- 두 실행의 `[검토]` 첫 줄(통과/반려)을 비교해 보세요. 만약 `메모 전달` 실행에서도 `반려`가 나왔다면 검토자가 지적한 누락 항목을 읽어 보세요. 고객 안내문에 굳이 다 넣지 않아도 되는 세부 번호까지 빠졌다고 지적했다면, 이는 작성자의 초안이 나빠서가 아니라 검토자의 판정 기준이 "조사 메모의 모든 사실을 빠짐없이 담았는가"로 너무 엄격하게 설정되어 있기 때문입니다. 다섯 줄 이내로 요약하라는 작성자 지침과 모든 항목을 포함하라는 검토자 지침이 충돌하면 대부분 반려가 발생합니다. 즉, 역할 간의 평가 기준 역시 전달물 계약의 일부로 맞춰 주어야 합니다. 또한 검토자는 원본 자료(`sources`)를 직접 보지 않고 조사 메모만 기준으로 삼기 때문에, 만약 조사 메모 자체가 틀렸다면 검토자도 이를 잡아내지 못한다는 점을 기억해 두세요.
- 한 번의 파이프라인 실행에서 모델 호출은 총 세 번(조사, 작성, 검토) 일어나며, 각 호출의 입력 토큰 합계가 전체 비용이 됩니다. 역할을 나누면 개별 호출의 입력 크기는 줄어들 수 있지만 호출 횟수는 늘어납니다. 표의 `입력 토큰`을 직접 더해 전체 비용을 확인해 보세요.

지금까지는 한 번에 하나의 노드만 차례대로 실행했습니다. 그런데 1절의 위상 정렬 결과에서 `묶음 2`에는 세 개의 작업(`billing`, `policy`, `outage`)이 함께 들어 있었습니다. 이 세 작업을 동시에 실행하면 무엇이 달라지는지 다음 절에서 살펴봅니다.

## 3절. 병렬, reducer, 최장 경로

**문제.** 1절에서 확인했듯이 `billing`, `policy`, `outage` 세 작업은 서로 상대방의 결과를 필요로 하지 않습니다. 그렇다면 세 작업을 동시에 실행해서 전체 대기 시간을 줄일 수 있을까요? 작업을 병렬로 실행하려면 코드에서 두 가지가 먼저 준비되어야 합니다. 첫째, 여러 작업이 동시에 같은 State 필드에 결과를 쓸 때 **어떻게 충돌 없이 합칠 것인가(reducer)**입니다. 둘째, 작업들이 정말로 동시에 실행되었는지 **어떻게 측정하고 검증할 것인가**입니다.

**먼저 예측하세요.** 코드를 실행하기 전에 아래 두 가지 질문에 대한 답을 먼저 예측해 보세요.

1. 세 개의 병렬 작업이 병합 규칙(reducer)이 없는 일반 문자열 필드 `last`에 동시에 값을 쓰면 어떻게 될까요? 마지막에 끝난 작업의 값이 덮어쓰여 남을까요, 아니면 프레임워크가 오류를 발생시킬까요?
2. 각각 0.3초가 걸리는 작업 세 개를 동시에 실행하면 전체 소요 시간은 약 몇 초일까요? 반대로 일반 `for` 반복문으로 차례대로 실행하면 몇 초가 걸릴까요?

### `Send`를 처음부터

지금까지 사용한 일반 간선(`add_edge`)은 그래프를 컴파일할 때 연결 대상이 미리 고정됩니다. 하지만 실행 시점에 들어온 계획(plan)의 길이에 따라 몇 개의 작업을 동시에 띄워야 할지 달라진다면 간선을 미리 고정해 그릴 수 없습니다. LangGraph에서는 **`Send`** 객체가 이 동적 병렬 분기(fan-out) 문제를 해결합니다.

`Send("노드 이름", 입력_데이터)`는 **"지정한 Node를, 이 개별 입력으로, 한 번 실행하라"**는 **작업 지시서 한 장**과 같습니다. 구현 방법은 다음 세 단계로 이루어집니다.

1. 조건부 간선에 등록한 분기 함수가 `Send` 객체들의 **리스트**를 반환합니다. 반환된 리스트의 길이가 곧 동시에 실행할 작업의 개수가 됩니다.
2. `add_conditional_edges`의 세 번째 인자에 도착 가능한 Node 이름 목록을 명시합니다. 그래프 시각화 도구(`get_graph`)가 이 목록을 보고 연결선을 그립니다.
3. `Send`를 받는 Node 함수의 첫 번째 인자는 그래프의 전체 State가 아니라 각 `Send`에 담아 보낸 **개별 입력 데이터(`payload`)**입니다. 덕분에 각 Worker에게 전체 상태 대신 그 작업에 꼭 필요한 최소 입력만 전달할 수 있습니다.

한 번의 분기에서 함께 반환된 `Send` 작업들은 그래프의 동일한 실행 단계(superstep) 안에서 동시에 실행되며, 그 단계의 작업들이 모두 끝나야 다음 단계의 Node로 넘어갑니다. 이때 각 작업이 반환한 결과는 공유 State에 기록됩니다. 바로 여기서 앞서 예측한 첫 번째 질문과 만납니다. 같은 실행 단계에서 여러 작업이 동일한 State 필드에 값을 쓰려고 하면 어떤 일이 벌어질까요?

아래 코드는 병합 규칙(reducer)이 없는 일반 필드에 `Send` 세 개가 동시에 값을 쓸 때 발생하는 예외와, 비교 기준이 되는 일반 `for` 반복문의 실행 시간을 차례로 확인합니다.

In [ ]:
# 실패 재현: 병합 규칙(reducer)이 없는 일반 필드(last)에 Send 세 개가 동시에 값을 쓰면 예외가 발생합니다.
class PlainState(TypedDict, total=False):
    plan: dict
    last: str

def fan_out_plain(state):
    # plan의 각 작업마다 write_last 노드를 실행하라는 Send 객체를 생성합니다.
    return [Send("write_last", {"task_id": name}) for name in state["plan"]]

def write_last(payload: dict):
    # 세 작업이 모두 동일한 'last' 필드에 자신의 task_id를 기록하려고 시도합니다.
    return {"last": payload["task_id"]}

builder = StateGraph(PlainState)
builder.add_node("write_last", write_last)
builder.add_conditional_edges(START, fan_out_plain, ["write_last"])
builder.add_edge("write_last", END)
plain_app = builder.compile()

try:
    plain_app.invoke({"plan": {"billing": 0.3, "policy": 0.3, "outage": 0.3}})
except InvalidUpdateError as error:
    print(f"{type(error).__name__}: {str(error).splitlines()[0]}")
print("모델 호출 0회")

In [ ]:
# 비교 기준: 0.3초가 걸리는 세 작업을 일반 for 반복문으로 순차 실행할 때의 전체 시간과 타임라인을 측정합니다.
WORK = {"billing": 0.3, "policy": 0.3, "outage": 0.3}

trace, started = Trace("for 문"), time.perf_counter()
for task_id, seconds in WORK.items():
    with trace.span(task_id):
        time.sleep(seconds)
print(f"for 문 순차 실행: 전체 {time.perf_counter() - started:.3f}초, 겹침 {overlap_seconds(trace.spans):.3f}초")
print(timeline(trace.spans))
print("모델 호출 0회")

위 실패 재현에서 발생한 `InvalidUpdateError`의 오류 메시지는 `At key 'last': Can receive only one value per step.`으로 시작합니다. 어느 상태 필드(`last`)에서 충돌이 일어났는지 정확히 짚어 주며, 기본 필드는 한 실행 단계(step)에서 하나의 값만 받을 수 있다고 알려 줍니다. 해결책은 해당 필드에 **여러 값을 하나로 합치는 규칙(reducer)**을 지정하는 것입니다. 파이썬 타입 힌트에서 `Annotated[타입, 병합_함수]` 형식으로 선언하며, 이 과정에서는 `merge_results` 함수를 사용합니다. `merge_results`는 작업 ID(`task_id`)를 키로 삼아 여러 dict 결과를 안전하게 병합합니다. 왜 단순히 리스트에 이어 붙이는(`operator.add`) 방식 대신 키 기반 병합 함수를 쓰는지 잠시 뒤에 실패 재현으로 직접 확인합니다.

두 번째 셀의 `for` 반복문 출력은 병렬 실행과 대조할 기준선(baseline)입니다. 타임라인 막대가 계단 모양으로 차례대로 이어진다면 동시 실행 겹침(overlap)이 전혀 없다는 뜻입니다. 실행 시간이 짧은 순차 반복문을 병렬 실행으로 착각하지 마세요. 타임라인에서 **세로로 겹치는 구간이 존재해야** 실제 동시 실행입니다.

이어서 작성할 Node 함수는 두 번째 인자로 `config: RunnableConfig`를 받습니다. 이는 그래프를 호출(`invoke`)할 때 외부에서 주입하는 설정값을 읽는 통로입니다. 측정 장치인 `Trace` 객체를 이 `config`로 넘기면, 그래프를 매번 새로 컴파일하지 않고도 실행할 때마다 독립된 새 측정 기록을 남길 수 있습니다. 이 구조가 바로 뒤이어 수행할 병렬 대 직렬 통제 비교의 핵심이 됩니다.

### 그림으로 보기: `Send`와 reducer

`Send`와 `reducer`를 결합해 동적 병렬 실행을 만드는 과정을 그림으로 나타내면 아래와 같습니다. 바로 다음 코드 셀에서 정의할 `dispatch`, `work`, `FanState`가 그림의 각 부분에 대응합니다.

![Send로 같은 Node를 세 번 실행하는 흐름. dispatch가 Send 세 개를 돌려주면 work가 입력만 달리해 세 번 실행되고, 한 단계가 끝나면 reducer가 붙은 results 칸에 합쳐진다](./assets/03a-3-send-fanout.png)

- `dispatch` 함수가 반환하는 것은 작업의 실행 결과가 아니라 **작업 지시서(`Send`) 세 장**입니다. 리스트의 길이가 곧 동시 실행할 작업의 수입니다.
- 세 개의 `work` 실행은 동일한 Node 함수에 입력(`payload`)만 다르게 주어 동시에 실행한 것입니다.
- 세 작업이 끝나면 모두 동일한 `results` 필드에 결과를 반환합니다. 앞서 본 `InvalidUpdateError`를 막기 위해 이 필드에 병합 규칙(`merge_results`)을 붙여 두었습니다.

아래 코드는 `Annotated[dict, merge_results]`가 적용된 병렬 그래프를 만들고, 동일한 그래프 객체에서 동시성 설정(`max_concurrency`)만 바꾸어 병렬과 직렬을 비교합니다.

In [ ]:
# results 필드에 merge_results 병합 규칙(reducer)을 지정하여 Send 병렬 실행 그래프를 구성합니다.
class FanState(TypedDict, total=False):
    plan: dict
    results: Annotated[dict, merge_results]

def dispatch(state):
    # 작업 계획(plan)의 각 항목마다 필요한 최소 입력(task_id, seconds)만 담아 Send 리스트를 반환합니다.
    return [Send("work", {"task_id": name, "seconds": seconds}) for name, seconds in state["plan"].items()]

def work(payload: dict, config: RunnableConfig):
    # config로 전달받은 Trace 객체에 실행 구간을 기록하며 지정된 시간만큼 대기합니다.
    with config["configurable"]["trace"].span(payload["task_id"]):
        time.sleep(payload["seconds"])
    return {"results": {payload["task_id"]: {"seconds": payload["seconds"]}}}

builder = StateGraph(FanState)
builder.add_node("work", work)
builder.add_conditional_edges(START, dispatch, ["work"])
builder.add_edge("work", END)
fan_app = builder.compile()

def run_fan(plan: dict, **options) -> dict:
    # 동일한 fan_app 그래프를 주어진 config 옵션으로 실행하고 소요 시간과 구간 기록을 반환합니다.
    trace, started = Trace("fan"), time.perf_counter()
    state = fan_app.invoke({"plan": plan}, config={"configurable": {"trace": trace}, **options})
    return {"seconds": round(time.perf_counter() - started, 3), "spans": trace.spans, "results": state["results"], "graph_id": id(fan_app), "options": options}

print("Send가 전달하는 개별 입력 (전체 plan이 아니라 해당 Worker에 필요한 최소 입력만 전달됩니다):")
for item in dispatch({"plan": WORK}):
    print(" ", item.node, item.arg)
show_graph(fan_app)

In [ ]:
# Send 병렬 그래프를 기본 설정(동시 실행)으로 실행하고 소요 시간과 겹침 구간을 확인합니다.
fan_parallel = run_fan(WORK)
print(f"Send 병렬 실행: 전체 {fan_parallel['seconds']:.3f}초, 겹침 {overlap_seconds(fan_parallel['spans']):.3f}초")
print(timeline(fan_parallel["spans"]))
print("병합된 결과:", fan_parallel["results"])
print("모델 호출 0회")

In [ ]:
# 동일한 그래프 객체(fan_app)에 max_concurrency=1 옵션만 주어 직렬 실행하고 병렬 실행과 비교합니다.
fan_serial = run_fan(WORK, max_concurrency=1)
show_table([
    {"실행": label, "config 차이": run["options"] or "없음", "plan_hash": plan_hash(WORK), "전체(초)": run["seconds"], "겹침(초)": overlap_seconds(run["spans"])}
    for label, run in [("병렬", fan_parallel), ("직렬", fan_serial)]
])
print(timeline(fan_serial["spans"]))
print("동일한 그래프 객체 여부:", fan_parallel["graph_id"] == fan_serial["graph_id"], "/ 그래프 코드는 전혀 수정하지 않았으며, 바뀐 것은 config의 max_concurrency 하나뿐입니다.")
print("모델 호출 0회")

LangGraph 설정의 `max_concurrency`는 한 실행 단계에서 동시에 돌아갈 수 있는 최대 작업 수를 제한합니다. 이 값을 `1`로 주면 세 개의 `Send` 작업이 한 번에 하나씩 차례대로 실행됩니다. 이 옵션 덕분에 **그래프 구조와 작업 분해를 완전히 동일하게 유지한 채 오직 동시성만 바꾼 통제 비교**가 가능해집니다. 표의 `plan_hash`는 작업 계획을 정렬된 JSON으로 직렬화해 해시한 값으로, 두 실행의 작업 분해가 완전히 같다는 것을 증명하는 지문 역할을 합니다. 만약 병렬용 그래프와 직렬용 그래프를 별도 코드로 따로 작성해 비교하면, 도중에 작업 수나 프롬프트 입력이 미묘하게 달라져도 이를 알아채기 어렵습니다.

이번에는 세 작업의 소요 시간을 균일하지 않게(`0.2초, 0.2초, 1.0초`) 바꾸어 실행해 봅니다. 작업을 병렬로 실행했을 때 걸리는 전체 시간은 개별 작업 시간의 합계가 아니라 **최장 경로(longest path)**, 즉 선행 의존 관계를 지키면서 이어지는 경로 중 가장 오래 걸리는 줄의 길이로 결정됩니다. `longest_path(시간_사전, 의존_사전)` 함수가 이 이론적 최장 경로를 계산합니다. 이론 계산값과 실제 측정값을 나란히 놓고 비교해 봅니다.

In [ ]:
# 작업 시간이 불균일할 때(0.2초, 0.2초, 1.0초) 최장 경로 계산값과 실제 병렬·직렬 실행 시간을 비교합니다.
UNEVEN = {"billing": 0.2, "policy": 0.2, "outage": 1.0}
uneven_parallel, uneven_serial = run_fan(UNEVEN), run_fan(UNEVEN, max_concurrency=1)
calculated = longest_path(UNEVEN, {})
show_table([
    {"실행": "병렬", "계산(초)": calculated, "계산식": "longest_path", "실측(초)": uneven_parallel["seconds"], "겹침(초)": overlap_seconds(uneven_parallel["spans"])},
    {"실행": "직렬", "계산(초)": round(sum(UNEVEN.values()), 3), "계산식": "합계", "실측(초)": uneven_serial["seconds"], "겹침(초)": overlap_seconds(uneven_serial["spans"])},
])
print(timeline(uneven_parallel["spans"]))
print("후속 작업 credit(0.1초)까지 연결했을 때의 최장 경로 계산:", longest_path({**UNEVEN, "credit": 0.1}, {"credit": list(UNEVEN)}))
print("모델 호출 0회")

### 그림으로 보기: 합계와 최장 경로

방금 출력한 표의 `계산(초)` 열이 아래 그림의 수식과 정확히 일치합니다. 그림에 적힌 숫자는 실행 측정값이 아니라 `UNEVEN`의 설정값으로 계산한 이론값이며, 실제 소요 시간은 위 표의 `실측(초)` 열에서 확인할 수 있습니다.

![작업 시간이 0.2, 0.2, 1.0초일 때 차례로 돌리면 합계 1.4초이고 동시에 돌리면 최장 경로 1.0초이며, credit이 셋을 기다리면 1.1초다](./assets/03a-4-longest-path.png)

- 작업을 차례대로 실행하면 전체 시간은 모든 작업 시간의 **합계(1.4초)**가 됩니다.
- 작업을 동시에 실행하면 가장 오래 걸리는 작업 하나(`outage`, 1.0초)가 전체 대기 시간을 결정합니다. 짧은 두 작업(`billing`, `policy`)을 병렬로 돌려서 아낄 수 있는 시간은 그 두 작업이 차지하던 시간(0.4초)뿐입니다.
- 만약 후속 작업인 `credit`(0.1초)이 세 작업의 결과를 모두 기다려야 한다면, 가장 늦게 끝나는 `outage` 뒤에 이어 붙어 총 **1.1초**가 됩니다. 선행 의존이 있는 작업은 앞 작업과 동시에 실행할 수 없으므로 시간이 그대로 더해집니다.

아래 코드에서는 선행 의존 관계를 무시하고 `credit`까지 한꺼번에 `Send`로 보냈을 때 어떤 조용한 오류(silent failure)가 발생하는지, 그리고 `merge_results`가 값 충돌을 어떻게 막아 주는지 확인합니다.

In [ ]:
# 실패 재현: 선행 의존을 무시하고 credit 작업까지 첫 단계에서 함께 Send로 실행합니다.
class FactState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    credit: int

def fact_work(payload: dict):
    # 전달받은 known 상태에서 앞선 조사 결과를 꺼내 계산에 사용합니다.
    known = {key: value for fact in payload["known"].values() for key, value in fact.items()}
    facts = {"billing": lambda: {"base_fee": load_billing("알파")[0]["base_fee_part"]}, "policy": lambda: {"percent": CREDIT_PERCENT},
             "outage": lambda: {"availability": load_outage("알파")["2026-08"]["availability_percent"]},
             "credit": lambda: {"credit": known.get("base_fee", 0) * known.get("percent", 0) // 100, "입력": known}}
    return {"results": {payload["task_id"]: facts[payload["task_id"]]()}}

def build_facts(fan: list[str], after: bool):
    # after=False이면 credit까지 한꺼번에 Send하고, after=True이면 세 조사 작업이 끝난 뒤 다음 단계에서 credit을 실행합니다.
    def send_all(state):
        return [Send("fact_work", {"task_id": name, "known": state.get("results", {})}) for name in fan]
    def credit_after(state):
        return fact_work({"task_id": "credit", "known": state["results"]})
    builder = StateGraph(FactState)
    builder.add_node("fact_work", fact_work)
    builder.add_conditional_edges(START, send_all, ["fact_work"])
    builder.add_node("credit_after", credit_after)
    builder.add_edge("fact_work", "credit_after" if after else END)
    builder.add_edge("credit_after", END)
    return builder.compile()

together = build_facts(["billing", "policy", "outage", "credit"], after=False).invoke({})
ordered = build_facts(["billing", "policy", "outage"], after=True).invoke({})
show_table([{"그래프": "함께 Send (의존 무시)", "credit": together["results"]["credit"]["credit"], "credit이 받은 입력": together["results"]["credit"]["입력"]},
            {"그래프": "Send 뒤에 순차 연결", "credit": ordered["results"]["credit"]["credit"], "credit이 받은 입력": ordered["results"]["credit"]["입력"]}])
print("모델 호출 0회")

In [ ]:
# 실패 재현: 동일한 작업 ID(outage)에 서로 다른 결과값이 들어오면 merge_results가 즉시 예외를 발생시킵니다.
first = {"outage": {"availability": 99.3}}
print("재전달(동일한 값):", merge_results(first, {"outage": {"availability": 99.3}}))
print("다른 작업 추가:", merge_results(first, {"billing": {"base_fee": 1000000}}))
try:
    merge_results(first, {"outage": {"availability": 99.95}})
except ValueError as error:
    print(f"상충(서로 다른 값): {type(error).__name__}: {error}")
print("모델 호출 0회")

**관찰할 것.** 여기까지의 실험은 모델을 호출하지 않는 일반 함수로 진행했기 때문에 실행할 때마다 결과가 일관되게 재현됩니다.

- 첫 번째 표에서 `credit이 받은 입력` 열을 확인해 보세요. 네 작업을 한꺼번에 `Send`한 경우 `credit`이 받은 입력은 빈 딕셔너리(`{}`)였고, 아무런 파이썬 예외도 발생하지 않은 채 `credit` 금액이 `0`으로 계산되었습니다. **`Send`는 작업 간의 선행 의존 관계를 자동으로 알지 못합니다.** 단지 넘겨받은 작업 지시서들을 동시에 실행할 뿐입니다. 따라서 선행 조건이 있는 작업은 반드시 그래프의 다음 단계(Edge)로 분리해야 합니다. 이러한 설계 오류는 런타임 예외를 내지 않고 조용히 `0원`을 반환하므로, 결과 수치를 검증하는 장치가 없으면 잘못된 답변이 그대로 고객에게 전달됩니다.
- 두 번째 셀에서 `merge_results`는 동일한 작업 ID에 같은 값이 다시 들어오는 경우(재전달)는 그대로 통과시키고, 서로 다른 값이 들어오는 경우(상충)는 `ValueError`로 즉시 멈춥니다. 분산 환경이나 에이전트 재시도 과정에서는 동일한 작업 결과가 중복 전달되는 일이 흔합니다. 만약 단순 리스트 추가(`operator.add`)를 reducer로 썼다면 중복된 결과가 두 번씩 쌓이고, 상충하는 두 값도 아무런 경고 없이 리스트에 나란히 남게 됩니다.
- 앞서 본 최장 경로 표에서 병렬 실행의 `실측(초)`이 이론적 `계산(초)`보다 약간 더 걸린 차이만큼이 그래프 런타임과 스레드 스케줄링의 기본 오버헤드입니다. 가장 오래 걸리는 작업 하나(`outage`, 1.0초)가 전체 응답 시간을 결정하는 상황이라면, 그 병목 작업을 줄이지 않고 나머지 짧은 작업들만 병렬화해서 얻을 수 있는 시간 이득이 얼마나 될지 항상 먼저 계산해 봐야 합니다.

이제 한 가지 명확한 가설을 세우고 실제 LLM Worker 실행을 통해 검증해 봅니다. **"병렬 실행이 줄여 주는 것은 전체 대기 시간(wall-clock time)뿐이며, 모델 호출 횟수와 토큰 합계는 줄어들지 않는다."** 단순 `time.sleep` 함수만으로는 토큰과 모델 호출을 측정할 수 없으므로, 이제 실제 모델을 사용하는 세 명의 Worker를 투입합니다.

아래 코드에서 정의하는 `PLAN`은 코드로 고정한 작업 분해입니다. 세 Worker(`billing`, `policy`, `outage`)를 동일한 입력과 동일한 그래프 객체(`plan_app`)에서 실행합니다. 두 실행 사이에 달라지는 것은 오직 `max_concurrency` 설정 하나뿐이며, 예산(budget)과 근거(Evidence) 기록이 서로 섞이지 않도록 각 실행마다 새로운 `Desk` 인스턴스를 생성해 전달합니다.

In [ ]:
# T-1001을 조사할 세 명의 실제 Worker 작업 계획(PLAN)과 Send 병렬 실행 그래프(plan_app)를 정의합니다.
PLAN = {"case_id": T1001["id"], "question": T1001["question"],
        "tasks": [{"task_id": role, "goal": f"{ROLE_GOALS[role]}\n고객 요청: {T1001['question']}", "allowed_tools": sorted(ROLE_TOOLS[role])} for role in ROLES]}

class PlanState(TypedDict, total=False):
    tasks: list[dict]
    results: Annotated[dict, merge_results]

def real_worker(payload: dict, config: RunnableConfig):
    # 전달받은 개별 작업 계약(WorkerTask)에 따라 실제 모델 Worker를 실행합니다.
    task = WorkerTask.model_validate(payload["task"])
    try:
        result = run_worker(task, config["configurable"]["desk"], model)
    except BudgetExceeded as error:  # 예산(budget) 초과가 발생하더라도 그래프가 멈추지 않고 실패 행으로 기록되게 합니다.
        result = WorkerResult(task_id=task.task_id, status="failed", error=str(error))
    return {"results": {task.task_id: result.model_dump()}}

def build_plan_app(worker):
    # tasks 목록의 각 작업을 Send로 동시에 내보내고 결과를 merge_results로 모으는 그래프를 컴파일합니다.
    builder = StateGraph(PlanState)
    builder.add_node("worker", worker)
    builder.add_conditional_edges(START, lambda state: [Send("worker", {"task": t}) for t in state["tasks"]], ["worker"])
    builder.add_edge("worker", END)
    return builder.compile()

plan_app = build_plan_app(real_worker)

def run_plan(plan: dict, label: str, app=plan_app, **options) -> dict:
    # 매 실행마다 독립된 Desk와 ModelMeter를 생성하여 지정된 동시성 옵션으로 그래프를 실행합니다.
    desk, meter, started = make_desk(T1001, label=label), ModelMeter(), time.perf_counter()
    state = app.invoke({"tasks": plan["tasks"]}, config={"callbacks": [meter], "configurable": {"desk": desk}, **options})
    return {"label": label, "plan_hash": plan_hash(plan), "results": state["results"], "desk": desk, "meter": meter, "seconds": round(time.perf_counter() - started, 3), "graph_id": id(app), "options": options}

def show_plan_run(run: dict) -> None:
    # 실행 시간, 겹침 시간, 타임라인 막대, 그리고 Worker별 결과 요약표를 출력합니다.
    print(f"[{run['label']}] plan_hash {run['plan_hash']}, 전체 {run['seconds']}초, 겹침 {overlap_seconds(run['desk'].trace.spans)}초")
    print(timeline(run["desk"].trace.spans))
    results = {key: WorkerResult.model_validate(value) for key, value in run["results"].items()}
    show_table(join_results(results, list(results))["rows"])

print("plan_hash:", plan_hash(PLAN), "/ 작업 목록:", [t["task_id"] for t in PLAN["tasks"]])

In [ ]:
# 실제 LLM Worker 3개를 Send를 통해 병렬(기본 설정)로 실행합니다.
announce("실제 Worker 3개를 Send로 병렬 실행", model_calls=12)
parallel_run = run_plan(PLAN, "병렬")
show_plan_run(parallel_run)

In [ ]:
# 동일한 PLAN과 동일한 그래프 객체를 max_concurrency=1로 설정하여 직렬 실행합니다.
announce("동일한 PLAN과 그래프를 max_concurrency=1로 직렬 실행", model_calls=12)
serial_run = run_plan(PLAN, "직렬", max_concurrency=1)
show_plan_run(serial_run)

In [ ]:
# 병렬 실행과 직렬 실행의 시간, 토큰 비용, 필수 사실 수집률을 한 표로 비교합니다.
from mas_lab.evaluation import evaluate_row, load_gold, routing_table

gold = load_gold()

def plan_row(run: dict) -> dict:
    totals, spans = run["meter"].totals(), run["desk"].trace.spans
    answer = " ".join(fact for result in run["results"].values() for fact in result["facts"])
    check = evaluate_row({"structure": run["label"], "case_id": T1001["id"], "answer": answer}, gold)
    required = len(gold[T1001["id"]]["required_any"])
    return {"실행": run["label"], "config 차이": run["options"] or "없음", "plan_hash": run["plan_hash"], "전체(초)": run["seconds"], "겹침(초)": overlap_seconds(spans),
            "합산(초)": round(sum(s["end"] - s["start"] for s in spans), 2), "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"],
            "출력 토큰": totals["output_tokens"], "usage 누락": totals["usage_missing"], "필수 사실": f"{required - len(check['누락 사실'])}/{required}", "누락 사실": check["누락 사실"]}

print("1회 실행 관찰 결과입니다(통계적 평균이 아닙니다).")
print("동일한 그래프 객체 여부:", parallel_run["graph_id"] == serial_run["graph_id"], "/ 동일한 plan_hash 여부:", parallel_run["plan_hash"] == serial_run["plan_hash"])
show_table([plan_row(parallel_run), plan_row(serial_run)])

**관찰할 것.** 위 비교표는 1회 실행 관찰 결과입니다. 앞서 세운 가설을 표의 각 열과 대조해 보세요.

- 먼저 두 행의 `plan_hash`가 완전히 동일한지 확인합니다. 작업 분해가 같아야 이어지는 시간 및 비용 비교가 의미를 가집니다.
- `전체(초)`와 `겹침(초)`를 함께 비교해 보세요. 직렬 실행(`max_concurrency=1`)의 `겹침(초)`는 반드시 `0`이어야 합니다. 만약 직렬 실행에서 겹침이 0보다 크다면 측정 코드를 의심해야 합니다.
- `합산(초)`은 세 Worker가 각각 실행된 시간 구간의 길이를 모두 더한 값입니다. 병렬 실행에서 `합산(초)`이 `전체(초)`보다 훨씬 크다면, 바로 그 차이만큼 세 Worker가 동시에 대기 시간을 겹쳐서 아낀 것입니다.
- 이제 `모델 호출` 횟수와 `입력 토큰`을 비교해 보세요. 병렬로 실행했다고 해서 모델 호출 횟수나 토큰이 줄어드나요? 만약 두 행의 호출 수나 토큰 수에 약간의 차이가 있다면, 그 원인은 동시성(`max_concurrency`) 때문이 아닙니다. 각 Worker가 내부에서 검색 도구를 몇 번 호출할지는 모델이 매 실행마다 결정하기 때문입니다. 어느 한쪽의 토큰이 조금 더 적게 나왔더라도 "병렬화(또는 직렬화)가 토큰을 줄였다"라고 해석해서는 안 됩니다. `max_concurrency`는 모델에 전달되는 프롬프트를 전혀 바꾸지 않습니다. 도구 호출이 한 번만 더 일어나도 그 호출의 전체 입력 토큰이 합산되므로, 호출 횟수가 달라진 두 실행은 토큰 수로 구조적 우열을 비교할 수 없습니다. 마찬가지 이유로 `전체(초)`의 차이에도 모델 호출 횟수 차이로 인한 변동이 일부 섞여 있을 수 있음을 감안해야 합니다. 앞서 수행한 `time.sleep` 함수 비교가 병렬화의 순수 시간 효과를 더 깔끔하게 보여 줄 수 있었던 이유가 바로 작업 내용이 고정되어 있었기 때문입니다.
- 각 실행에서 출력된 Worker 결과 요약표의 `상태`와 `근거 수`도 비교해 보세요. 동일한 입력을 주었는데도 세부 결과가 조금씩 다르다면 이것이 바로 LLM 실행의 자연스러운 변동성입니다. 만약 `상태`가 `complete`인데 `근거 수`가 `0`인 행이 있다면, 출력 스키마(`WorkerResult`)는 통과했지만 정식 근거(Evidence) 없이 완료를 주장한 것입니다. 현재 그래프는 그러한 결과도 그대로 병합합니다. 이를 검증하고 걸러내는 방법은 다음 노트북(03b, 04)에서 다룹니다.
- 마지막 열의 `필수 사실`과 `누락 사실`은 세 Worker가 수집한 사실(`facts`) 문자열만 모아서 정답 기준과 자동 대조한 결과입니다. 현재 `PLAN`에는 조사 Worker 세 개만 있고 크레딧 금액을 곱해 계산하는 후속 단계가 없으므로, 최종 계산 금액(예: `150,000`) 하나가 누락되는 것은 계획 자체가 조사 단계까지만 포함하고 있기 때문입니다. 만약 다른 기본 사실까지 누락되었다면 표현 방식의 차이인지 실제 조사 누락인지 위 Worker 요약표와 대조해 보세요.

02 노트북의 Supervisor에서는 한 번의 메시지에 `task`를 여러 개 담아 병렬로 위임할지 여부를 **부모 모델**이 결정했습니다. 반면 방금 실행한 그래프에서는 `Send`를 통해 **코드**가 세 작업을 동시에 내보내도록 고정했습니다. 병렬 실행을 프롬프트 지침으로 부탁하는 것과 그래프 구조로 고정하는 것은 전혀 다른 설계입니다.

그렇다고 해서 `Send` 자체가 곧 특정 Multi-Agent 아키텍처 패턴을 뜻하는 것은 아닙니다. `Send`는 입력 목록을 동적으로 나누어 실행하는 그래프 도구일 뿐이며, 그 도착 노드는 앞서 본 `time.sleep` 일반 함수일 수도 있고 독립된 Deep Agent Worker일 수도 있습니다. 우리가 일반 함수로 먼저 원리를 확인한 뒤 실제 Worker를 연결한 이유가 여기에 있습니다.

이제 다음 노트북(03b)에서는 **"요청에 따라 어느 경로로 보낼 것인가"**를 묻습니다. 만약 모든 고객 문의가 T-1001 같은 복합 문의라면 언제나 세 Worker를 모두 띄우면 되겠지만, "서비스 크레딧 신청 기한이 언제인가요?" 같은 단순 문의에까지 매번 세 Worker를 모두 실행할 필요는 없기 때문입니다.

## 직접 해보기

본인의 실제 업무나 새로운 조건으로 그래프 설정을 바꾸어 확인해 보는 실습입니다. 실제 모델을 호출하는 **해보기 3**만 `RUN_EXERCISE = False`로 꺼져 있으며, 일반 함수로 동작하는 **해보기 1**과 **해보기 2**는 값을 수정한 뒤 그대로 실행할 수 있습니다.

### 해보기 1

본인의 업무(또는 익숙한 도메인 업무)를 1절의 `TASKS` 사전 형식으로 정의하고, `plan_batches`를 통해 동시에 실행 가능한 작업 묶음을 찾아보세요. 아래 기본 예시는 '월간 운영 보고서 작성' 업무입니다. 각 작업의 `needs`(선행 작업) 목록을 바꾸어 보면서 실행 묶음이 어떻게 달라지는지 확인해 보세요. 또한 의도적으로 순환 의존이나 존재하지 않는 작업 이름을 넣어 `ValueError` 거절 메시지가 정확히 작동하는지도 확인해 보세요.

In [ ]:
# 본인의 업무를 작업 사전(MY_TASKS)으로 정의하고 위상 정렬 묶음을 확인합니다.
MY_TASKS = {
    "수집": {"needs": [], "state": "raw_data", "done": "이번 달 원시 자료가 모임"},
    "매출 분석": {"needs": ["수집"], "state": "sales", "done": "전월 대비 매출 증감을 비교함"},
    "장애 분석": {"needs": ["수집"], "state": "incidents", "done": "주요 장애 원인을 정리함"},
    "초안": {"needs": ["매출 분석", "장애 분석"], "state": "draft", "done": "요약 표와 본문이 작성됨"},
    "승인": {"needs": ["초안"], "state": "approved", "done": "담당자가 최종 승인함"},
}
for number, batch in enumerate(plan_batches(MY_TASKS), 1):
    print(f"묶음 {number}: {batch}")
print("모델 호출 0회")

### 해보기 2

병렬 실행의 시간 단축 이득이 거의 사라지는 작업 시간 분포를 직접 찾아보세요. 아래 코드의 `MY_WORK` 시간 값들을 바꾸어 가며 표의 `이득(초)`과 `이득 비율`을 확인하고, 왜 병렬화 효과가 작아지는지 **최장 경로(longest path)** 개념으로 설명해 보세요. 특히 전체 작업 중 한 작업이 합계 시간의 대부분을 차지할 때 병렬화 이득이 어떻게 변하는지 확인해 보세요.

In [ ]:
# 세 작업의 소요 시간 비율(MY_WORK)을 바꾸어 가며 최장 경로와 병렬 실행 이득을 비교합니다.
MY_WORK = {"billing": 0.05, "policy": 0.05, "outage": 0.6}  # 각 작업의 시간(초)을 바꾸어 실행해 보세요.
parallel, serial = run_fan(MY_WORK), run_fan(MY_WORK, max_concurrency=1)
show_table([{"plan_hash": plan_hash(MY_WORK), "최장 경로(초)": longest_path(MY_WORK, {}), "병렬 실측(초)": parallel["seconds"], "직렬 실측(초)": serial["seconds"],
             "이득(초)": round(serial["seconds"] - parallel["seconds"], 3), "이득 비율": round(1 - parallel["seconds"] / serial["seconds"], 2)}])
print("모델 호출 0회")

### 해보기 3

(실제 모델 호출) 세 개로 나누었던 조사 작업을 두 개로 합쳤을 때도 필수 사실 수집률이 그대로 유지되는지 실험해 봅니다. 아래 코드의 `MERGED` 설정은 `billing`과 `outage` 조사를 하나의 Worker(`facts`)에게 함께 맡기고, `policy` 조사만 별도 Worker로 둡니다. 작업 묶음 구성을 바꾸어 본 뒤 `RUN_EXERCISE`를 `True`로 설정해 실행하고, 앞서 실행한 3-Worker 병렬 결과(`parallel_run`)와 `필수 사실`, `모델 호출`, `입력 토큰`을 비교해 보세요. 한 Worker에게 두 역할의 도구를 함께 주면 총 모델 호출 횟수가 줄어드나요, 아니면 단일 Worker의 Context가 커지나요?

In [ ]:
RUN_EXERCISE = False
MERGED = {"facts": ["billing", "outage"], "policy": ["policy"]}  # 역할을 묶는 방식을 바꾸어 실험해 보세요.

def merged_worker(payload: dict, config: RunnableConfig):
    # 합쳐진 역할의 허용 도구와 목표를 받아 실행하는 Worker 노드입니다.
    desk, task = config["configurable"]["desk"], WorkerTask.model_validate(payload["task"])
    allowed = frozenset(task.allowed_tools)
    agent = create_deep_agent(model=model, tools=[t for t in desk.tools("all") if t.name in allowed], system_prompt="주어진 작업 계약을 따르는 조사 Worker입니다.",
                              middleware=allowlist_middleware(desk.budget, allowed), permissions=DENY_WRITES, response_format=ProviderStrategy(WorkerResult), name=f"worker-{task.task_id}")
    with desk.trace.span(task.task_id, "worker"):
        state = agent.invoke({"messages": [{"role": "user", "content": contract_prompt(task, desk.scope)}]}, config={"recursion_limit": 40})
    return {"results": {task.task_id: state["structured_response"].model_copy(update={"task_id": task.task_id}).model_dump()}}

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 값을 수정한 뒤 RUN_EXERCISE를 True로 바꾸어 실행해 보세요.")
else:
    announce("작업을 합친 계획(MERGED)으로 병렬 실행", model_calls=8)
    goal = lambda roles: "\n".join(ROLE_GOALS[role] for role in roles) + f"\n고객 요청: {T1001['question']}"
    merged_plan = {**PLAN, "tasks": [{"task_id": name, "goal": goal(roles), "allowed_tools": sorted(frozenset().union(*(ROLE_TOOLS[r] for r in roles)))} for name, roles in MERGED.items()]}
    merged_run = run_plan(merged_plan, "합친 계획", app=build_plan_app(merged_worker))
    show_plan_run(merged_run)
    show_table([plan_row(parallel_run), plan_row(merged_run)])

## 정리

| 제어 흐름 | LangGraph 핵심 도구 | 방지하는 실패 | 실습에서 재현한 실패 |
|---|---|---|---|
| **순차 (Sequence)** | `add_edge`, 역할 간 전달물(`notes`) | 선행 결과 없이 뒷단계가 먼저 실행되는 문제 | 지원 범위 밖 고객("감마")에게도 멈추지 않고 `credit` 0원을 계산함 |
| **병렬 (Parallel)** | `Send`, `Annotated[..., reducer]`, `max_concurrency` | 서로 독립적인 작업들이 불필요하게 차례로 대기하는 문제 | reducer 누락 시 `InvalidUpdateError`, 선행 의존 무시 시 빈 입력(`{}`)으로 0원 계산 |

병렬 실행의 시간 단축 효과는 반드시 **동일한 작업 분해(`plan_hash` 일치)** 안에서만 비교해야 합니다. 이 노트북에서 동일한 계획과 동일한 그래프 객체를 두고 오직 `max_concurrency` 스위치만 바꾸어 비교한 이유가 바로 여기에 있습니다. 또한 병렬화가 줄여 주는 것은 전체 대기 시간(최장 경로로의 압축)일 뿐이며, 총 모델 호출 횟수나 토큰 비용이 줄어든다고 기대할 근거는 없습니다.

지금까지 다룬 순차와 병렬은 **업무의 선행 조건 표만으로** 실행 전에 경로가 결정됩니다. 그런데 만약 어떤 작업을 실행할지가 고객 요청의 난이도에 따라 달라지거나, 조사 결과가 부족할 때 다시 검색해야 한다면 정적인 선행 조건 표만으로는 그래프를 그릴 수 없습니다. 바로 그 두 가지 동적 흐름이 다음 노트북 **03b**에서 다룰 **분기(Branching)**와 **루프(Loop)**입니다.

## 확인 질문

1. `Send`를 사용해 여러 노드로 동적 병렬 분기(fan-out)를 수행할 때 결과를 받는 State 필드에 병합 규칙(reducer)이 반드시 필요한 이유는 무엇인가요? 3절의 실패 재현에서 확인한 `InvalidUpdateError` 오류 메시지와 연결하여 설명해 보세요.
2. 일반 함수(`time.sleep`)로 실행한 `max_concurrency=1` 비교 실험이 실제 LLM Worker 3개를 사용한 병렬·직렬 비교표보다 "병렬화의 순수 시간 단축 효과"를 더 명확하게 증명할 수 있는 이유는 무엇인가요?
3. 작업을 병렬로 실행해도 전체 입력 토큰 합계가 줄어들지 않는 이유는 무엇이며, 그럼에도 실무에서 병렬 구조를 선택해야 하는 조건은 언제인가요?

다음 노트북 **03b**에서는 **분기와 루프**를 직접 구현합니다. 고객 요청의 성격에 따라 서로 다른 경로(Fast/Slow Path)로 보내고, 근거가 부족하면 이전 단계로 되돌아가는 그래프에서 어떤 새로운 실패가 발생할 수 있는지 살펴봅니다.